# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. El dataset cardiovascular es únicamente didáctico: el resultado
no puede usarse para diagnóstico, tratamiento ni decisiones sobre pacientes.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos seis runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.pkl` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
import mlflow
import sklearn
import pandas as pd
import matplotlib

print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import pickle
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

STUDENT_ALIAS = "Alvaro"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
MIN_VALIDATION_RECALL = 0.72
DATASET_PATH = "../../../data/raw/heart.csv"

# TODO 1: valida el alias con una expresión regular y configura el tracking de Databricks.
if not re.fullmatch(r"[A-Za-z0-9_-]+", STUDENT_ALIAS):
    raise ValueError(
        "STUDENT_ALIAS debe contener solo letras minúsculas, números, '-' o '_'."
    )

mlflow.set_tracking_uri("databricks")

## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío o el target no es binario. Construye un `quality_report` serializable como JSON.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "target"
EXPECTED_FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]

# TODO 2: valida el contrato y crea X, y y quality_report.
# Debe incluir filas, duplicados, distribución del target, nulos y checks ejecutados.
X = data[EXPECTED_FEATURES]
y = data[TARGET]

quality_report = {
    "rows": len(data),
    "duplicates": data.duplicated().sum(),
    "target_distribution": y.value_counts().to_dict(),
    "nulls": data.isnull().sum().to_dict(),
    "checks_executed": ["columnas", "duplicados", "target", "nulos"]
}

print(quality_report)

## 3. Divide 60 % train, 20 % validación y 20 % test, siempre estratificado

In [0]:
# TODO 3: crea X_train, X_valid, X_test, y_train, y_valid e y_test.
# No uses X_test ni y_test hasta la sección 9.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validación:", X_valid.shape)
print("Test:", X_test.shape)

## 4. Pipeline y funciones de evaluación

In [0]:
categorical_columns = X.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_columns = [c for c in EXPECTED_FEATURES if c not in categorical_columns]


def build_pipeline(config: dict) -> Pipeline:
    # TODO 4a: ColumnTransformer con imputación y OneHotEncoder, seguido del RandomForest.
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns)
    ])

    model = RandomForestClassifier(**config)

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    # TODO 4b: accuracy, precision, recall, F1, ROC AUC y latencia por fila.
    start = time.perf_counter()

    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]

    elapsed = time.perf_counter() - start

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "roc_auc": roc_auc_score(labels, probabilities),
        "latency_ms_per_row": (elapsed / len(features)) * 1000
    }

## 5. Diseña al menos seis candidatos comparables

In [0]:
# TODO 5: cambia de forma controlada n_estimators, max_depth, min_samples_leaf
# y/o class_weight. Todos deben incluir random_state.
CANDIDATES = [
    {
        "n_estimators": 100,
        "max_depth": 5,
        "min_samples_leaf": 1,
        "random_state": RANDOM_STATE
    },
    {
        "n_estimators": 200,
        "max_depth": 5,
        "min_samples_leaf": 1,
        "random_state": RANDOM_STATE
    },
    {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 1,
        "random_state": RANDOM_STATE
    },
    {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 2,
        "random_state": RANDOM_STATE
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 2,
        "random_state": RANDOM_STATE
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 2,
        "class_weight": "balanced",
        "random_state": RANDOM_STATE
    }
]
assert len(CANDIDATES) >= 6, "Necesitas al menos seis candidatos."

## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.pkl` y contrato de inferencia.

In [0]:
dataset_card = {
    "source": "TODO",
    "purpose": "TODO",
    "features": EXPECTED_FEATURES,
    "target": TARGET,
    "split": {"train": 0.60, "validation": 0.20, "test": 0.20, "random_state": RANDOM_STATE},
    "known_limitations": ["TODO", "TODO", "TODO"],
}
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R2", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R3", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R4", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
    ],
}
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:
# TODO 6a: crea train_dataset y validation_dataset con mlflow.data.from_pandas.
# Incluye target alineado con cada split y usa DATASET_PATH como source.
train_data = X_train.copy()
train_data[TARGET] = y_train

validation_data = X_valid.copy()
validation_data[TARGET] = y_valid

train_dataset = mlflow.data.from_pandas(
    train_data,
    source=DATASET_PATH,
    targets=TARGET,
    name="train"
)

validation_dataset = mlflow.data.from_pandas(
    validation_data,
    source=DATASET_PATH,
    targets=TARGET,
    name="validation"
)

print(train_dataset.name)
print(validation_dataset.name)

### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, pickle,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    # TODO 6b: log_model(name='model'), tag candidate.model_uri, model.pkl,             ME HE AYUDADO CON LA IA PARA ESTE APARTADO
    # pickle_size_bytes, confusion matrix e inference_contract.json. Devuelve model_uri.
    input_example = X_train.head(3)

    signature = infer_signature(
        X_train,
        model.predict(X_train)
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=input_example
    )

    model_uri = model_info.model_uri

    mlflow.set_tag("candidate.model_uri", model_uri)

    # Guardar pickle
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "model.pkl"

        with open(path, "wb") as f:
            pickle.dump(model, f)

        mlflow.log_artifact(str(path), artifact_path="deployment")
        mlflow.log_metric("pickle_size_bytes", path.stat().st_size)

    # Matriz de confusión
    fig, ax = plt.subplots()

    ConfusionMatrixDisplay.from_predictions(
        y_valid,
        predictions,
        ax=ax
    )

    mlflow.log_figure(
        fig,
        "validation/confusion_matrix.png"
    )

    plt.close(fig)

    # Contrato de inferencia
    contract = {
        "features": EXPECTED_FEATURES,
        "target": TARGET
    }

    mlflow.log_dict(
        contract,
        "deployment/inference_contract.json"
    )

    return model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:
experiment_id = None
candidate_run_ids = []

# TODO 6c: implementa un start_run por configuración, registra tags/params/inputs/JSON,
# entrena, calcula validation.*, classification_report y llama log_candidate_artifacts.
for i, config in enumerate(CANDIDATES):

    with mlflow.start_run() as run:

        if experiment_id is None:
            experiment_id = run.info.experiment_id

        candidate_run_ids.append(run.info.run_id)

        mlflow.set_tags({
            "student_alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "run_type": "candidate"
        })

        mlflow.log_params(config)

        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(validation_dataset, context="validation")

        mlflow.log_dict(
            dataset_card,
            "governance/dataset_card.json"
        )

        mlflow.log_dict(
            quality_report,
            "governance/quality_report.json"
        )

        mlflow.log_dict(
            risk_register,
            "governance/risk_register.json"
        )

        model = build_pipeline(config)

        model.fit(X_train, y_train)

        metrics = evaluate_classifier(
            model,
            X_valid,
            y_valid
        )

        mlflow.log_metrics({
            f"validation.{k}": v
            for k, v in metrics.items()
        })

        predictions = model.predict(X_valid)

        report = classification_report(
            y_valid,
            predictions,
            output_dict=True
        )

        mlflow.log_dict(
            report,
            "validation/classification_report.json"
        )

        log_candidate_artifacts(
            model,
            predictions,
            i
        )
assert len(candidate_run_ids) == len(CANDIDATES)

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
# TODO 7: usa mlflow.search_runs con tres tags en filter_string y muestra parámetros,
# métricas de validación, latencia y run_id en un DataFrame `comparison`.
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=(
        f"tags.student_alias = '{STUDENT_ALIAS}' "
        f"AND tags.batch_id = '{BATCH_ID}' "
        f"AND tags.run_type = 'candidate'"
    )
)

comparison = runs[[
    "run_id",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_leaf",
    "metrics.validation.accuracy",
    "metrics.validation.precision",
    "metrics.validation.recall",
    "metrics.validation.f1",
    "metrics.validation.roc_auc",
    "metrics.validation.latency_ms_per_row"
]]

display(comparison)

## 8. Aplica el gate sin mirar test

Filtra por recall mínimo; ordena por F1 DESC, ROC AUC DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
# TODO 8: aplica el gate. Si no hay elegibles, falla sin rebajar el umbral a posteriori.
eligible = comparison[
    comparison["metrics.validation.recall"] >= MIN_VALIDATION_RECALL
].copy()

if eligible.empty:
    raise ValueError("Ningún modelo supera el recall mínimo")

eligible = eligible.sort_values(
    by=[
        "metrics.validation.f1",
        "metrics.validation.roc_auc",
        "metrics.validation.latency_ms_per_row"
    ],
    ascending=[False, False, True]
)

best = eligible.iloc[0]

BEST_RUN_ID = best["run_id"]

BEST_MODEL_URI = mlflow.get_run(
    BEST_RUN_ID
).data.tags["candidate.model_uri"]

selection_report = {
    "best_run_id": BEST_RUN_ID,
    "best_model_uri": BEST_MODEL_URI,
    "minimum_recall": MIN_VALIDATION_RECALL,
    "test_used_for_selection": False
}

print(selection_report)

## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
# TODO 9: load_model(BEST_MODEL_URI), calcula test.*, reabre BEST_RUN_ID,
# registra el input de testing, métricas, tags de selección y selection/decision.json.
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)

test_metrics = evaluate_classifier(
    best_model,
    X_test,
    y_test
)

test_data = X_test.copy()
test_data[TARGET] = y_test

test_dataset = mlflow.data.from_pandas(
    test_data,
    source=DATASET_PATH,
    targets=TARGET,
    name="test"
)

with mlflow.start_run(run_id=BEST_RUN_ID):

    mlflow.log_input(
        test_dataset,
        context="testing"
    )

    mlflow.log_metrics({
        f"test.{k}": v
        for k, v in test_metrics.items()
    })

    mlflow.set_tag("selection.winner", "true")
    mlflow.set_tag("selection.test_used", "false")

    mlflow.log_dict(
        selection_report,
        "selection/decision.json"
    )

print(test_metrics)

## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
# TODO 10: set_registry_uri('databricks-uc'), register_model, descripciones,
# tags, Challenger, promoción, rollback, re-promoción y variable champion.
mlflow.set_registry_uri("databricks-uc")
catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = spark.sql("SELECT current_schema()").first()[0]

safe_alias = re.sub(r"[^a-z0-9_]", "_", STUDENT_ALIAS.lower())

MODEL_NAME = f"{catalog}.{schema}.heart_model_{safe_alias}"

print(MODEL_NAME)

## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
# TODO 11: prueba que el consumidor puede cargar por alias y predecir tres filas.
champion_uri = f"models:/{MODEL_NAME}@Champion"

champion_model = mlflow.sklearn.load_model(
    champion_uri
)

smoke_predictions = champion_model.predict(
    X_test.head(3)
)

print(smoke_predictions)

## 12. API local con el pickle ganador

Resuelve el run de `Champion`, descarga `deployment/model.pkl` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
# TODO 12a: resuelve champion.run_id, descarga runs:/.../deployment/model.pkl,
# deserializa api_model y comprueba que predice una fila.
champion = client.get_model_version_by_alias(
    MODEL_NAME,
    "Champion"
)

champion_run_id = champion.run_id

### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

# TODO 12b: define PredictionHandler y actualiza API_STATS en cada respuesta.
API_STATS = {
    "requests_total": 0,
    "requests_success": 0,
    "requests_rejected": 0,
    "requests_error": 0,
    "latencies_ms": []
}

API_STATS_LOCK = threading.Lock()


class PredictionHandler(BaseHTTPRequestHandler):

    def _send_json(self, status, data):

        body = json.dumps(data).encode()

        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.end_headers()

        self.wfile.write(body)

    def do_GET(self):

        start = time.perf_counter()

        if self.path == "/health":

            self._send_json(200, {"status": "ok"})

            status = "success"

        else:

            self._send_json(404, {"error": "not found"})

            status = "rejected"

        latency = (time.perf_counter() - start) * 1000

        with API_STATS_LOCK:

            API_STATS["requests_total"] += 1
            API_STATS["latencies_ms"].append(latency)

            if status == "success":
                API_STATS["requests_success"] += 1
            else:
                API_STATS["requests_rejected"] += 1

    def do_POST(self):

        start = time.perf_counter()

        try:

            length = int(self.headers["Content-Length"])
            body = json.loads(self.rfile.read(length))

            if not isinstance(body, dict):
                raise ValueError("El JSON debe ser un objeto")

            instances = body.get("instances")

            if not isinstance(instances, list) or len(instances) == 0:
                raise ValueError("instances debe ser una lista no vacía")

            if not all(isinstance(x, dict) for x in instances):
                raise ValueError("Cada instancia debe ser un objeto")

            for instance in instances:

                if set(instance.keys()) != set(EXPECTED_FEATURES):
                    raise ValueError("Columnas incorrectas")

            df = pd.DataFrame(instances)

            predictions = api_model.predict(df)

            self._send_json(
                200,
                {"predictions": predictions.tolist()}
            )

            result = "success"

        except ValueError as e:

            self._send_json(
                400,
                {"error": str(e)}
            )

            result = "rejected"

        except Exception as e:

            self._send_json(
                500,
                {"error": str(e)}
            )

            result = "error"

        latency = (time.perf_counter() - start) * 1000

        with API_STATS_LOCK:

            API_STATS["requests_total"] += 1
            API_STATS["latencies_ms"].append(latency)

            if result == "success":
                API_STATS["requests_success"] += 1

            elif result == "rejected":
                API_STATS["requests_rejected"] += 1

            else:
                API_STATS["requests_error"] += 1

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:
# TODO 12c: crea api_server, API_PORT, api_thread, API_STOPPED y stop_api().
api_server = ThreadingHTTPServer(
    ("127.0.0.1", 0),
    PredictionHandler
)

API_PORT = api_server.server_address[1]

API_STOPPED = False

api_thread = threading.Thread(
    target=api_server.serve_forever,
    daemon=True
)

api_thread.start()


def stop_api():

    global API_STOPPED

    if not API_STOPPED:

        api_server.shutdown()
        api_server.server_close()

        api_thread.join(timeout=5)

        API_STOPPED = True


print("API puerto:", API_PORT)

## 13. Pruebas de contrato y de error

In [0]:
# TODO 13: crea call_json con try/finally para apagar siempre la API y verifica:
# GET /health -> 200; POST válido -> 200; sólo age -> 400; JSON no-object -> 400.
def call_json(method, path, payload=None):

    url = f"http://127.0.0.1:{API_PORT}{path}"

    data = None

    if payload is not None:
        data = json.dumps(payload).encode()

    request = Request(
        url,
        data=data,
        method=method,
        headers={"Content-Type": "application/json"}
    )

    try:

        with urlopen(request) as response:

            return (
                response.status,
                json.loads(response.read())
            )

    except HTTPError as e:

        return (
            e.code,
            json.loads(e.read())
        )

## 14. Registra un run separado para la evidencia del despliegue

In [0]:
# TODO 14: registra versión/alias/run origen, requests totales, éxitos, rechazadas,
# latencias y deployment/evidence.json. Conserva DEPLOYMENT_RUN_ID.
# No registres el payload de entrada: podría contener información sensible.
latencies = API_STATS["latencies_ms"]

with mlflow.start_run() as deployment_run:

    DEPLOYMENT_RUN_ID = deployment_run.info.run_id

    mlflow.set_tags({
        "student_alias": STUDENT_ALIAS,
        "batch_id": BATCH_ID,
        "run_type": "deployment",
        "source_run_id": BEST_RUN_ID,
        "model_alias": "Champion"
    })

    mlflow.log_metric(
        "requests_total",
        API_STATS["requests_total"]
    )

    mlflow.log_metric(
        "requests_success",
        API_STATS["requests_success"]
    )

    mlflow.log_metric(
        "requests_rejected",
        API_STATS["requests_rejected"]
    )

    if latencies:

        mlflow.log_metric(
            "latency_mean_ms",
            float(np.mean(latencies))
        )

        mlflow.log_metric(
            "latency_max_ms",
            float(np.max(latencies))
        )

    evidence = {
        "model": MODEL_NAME,
        "version": champion.version,
        "alias": "Champion",
        "source_run_id": BEST_RUN_ID,
        "api_stats": API_STATS
    }

    mlflow.log_dict(
        evidence,
        "deployment/evidence.json"
    )

## 15. Apaga la API y entrega el resumen del ciclo

In [0]:
# TODO 15: shutdown, server_close, join y assert de que el hilo terminó.
# Imprime experiment_id, batch, ganador, modelo/versión/alias y deployment run.
stop_api()

assert not api_thread.is_alive()

print("Experiment ID:", experiment_id)
print("Batch:", BATCH_ID)
print("Ganador:", BEST_RUN_ID)
print("Modelo:", MODEL_NAME)
print("Versión:", champion.version)
print("Alias:", "Champion")
print("Deployment run:", DEPLOYMENT_RUN_ID)